# S09 · When accuracy lies — judging any classifier honestly

The three classifiers from the first notebook all report a category — but how do
we *trust* any of them? Here is the trap: real fraud is rare. Almost every UPI
transaction is genuine, and on data that lopsided a model that does nothing can
look 95% accurate. We build a fraud-style dataset, watch that trap spring, and
then judge a classifier the way industry really does: the four counts,
precision, recall, F1 and ROC-AUC. Then we move the decision line and watch the
trade-off — on synthetic fraud data and on real breast-cancer data.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on
  each cell, top to bottom, and read the plain-English note above each one.
- New to the four counts? Open the primer
  `primers/reading_a_confusion_matrix.md` for a ten-minute, picture-first
  version. It is the heart of today.
- Already confident with code or with these metrics? Look for the cells marked
  **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses numpy, matplotlib and scikit-learn.
# Google Colab already ships all three, so there is nothing to install.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                                   # fast maths on numbers
import matplotlib.pyplot as plt                       # drawing charts
from sklearn.datasets import make_classification      # to invent a fraud-style dataset
from sklearn.model_selection import train_test_split  # to split train vs test
from sklearn.linear_model import LogisticRegression   # our classifier

## Step 1 — make an imbalanced (fraud-style) dataset

Fraud is rare: almost every transaction is legitimate. We invent 2,000 UPI
transactions where about **95%** are class 0 (legit) and only **5%** are class 1
(fraud). The `weights` setting controls that ratio. This lopsidedness is the
whole point of the notebook.

In [ ]:
# Set a seed so the dataset is the same every time.
np.random.seed(0)

# 2000 transactions, each described by a few numbers.
# weights=[0.95, 0.05] means 95% legit and 5% fraud (the rare class).
transactions, label = make_classification(
    n_samples=2000,
    n_features=6,
    n_informative=4,
    weights=[0.95, 0.05],
    random_state=0,
)

number_of_legit = np.sum(label == 0)
number_of_fraud = np.sum(label == 1)
print("legit (class 0):", number_of_legit)
print("fraud (class 1):", number_of_fraud)
print("fraud is only", round(100 * number_of_fraud / len(label), 1), "% of the data")

## Step 2 — the accuracy trap

Before training anything, look at a silly "model" that **always predicts
legit**. Because fraud is so rare, this lazy guess is right most of the time, so
its accuracy looks great. Yet it catches **zero** fraud. This is the **accuracy
paradox**, and it is why accuracy alone can fool you.

In [ ]:
# The lazy model: predict class 0 (legit) for every single transaction.
always_legit = np.zeros(len(label))

accuracy_of_lazy_model = np.mean(always_legit == label)
frauds_caught_by_lazy_model = np.sum((always_legit == 1) & (label == 1))

print("Accuracy of 'always predict legit':", round(accuracy_of_lazy_model, 3))
print("Frauds it caught                  :", int(frauds_caught_by_lazy_model))
print("High accuracy, completely useless. This is the accuracy paradox.")

## Step 3 — split into a training part and a test part

We train on one part of the data and judge on another part the model has never
seen, exactly like the train/test habit from Session 7. We add `stratify=label`
so both parts keep the same 95/5 ratio. That matters a lot when one class is
rare: without it, an unlucky split could leave almost no fraud in the test
part.

In [ ]:
# 70% for training, 30% kept aside for an honest test.
train_x, test_x, train_label, test_label = train_test_split(
    transactions, label,
    test_size=0.3,
    stratify=label,     # keep the same class ratio in both parts
    random_state=0,
)

print("training transactions :", train_x.shape[0])
print("test transactions     :", test_x.shape[0])
print("frauds in the test set:", int(np.sum(test_label == 1)))

## Step 4 — fit the classifier

Same model as the first notebook: logistic regression. It learns from the
training part only, then we ask it to predict on the hidden test part.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(train_x, train_label)

# Ask the model for its yes/no call on the hidden test transactions.
predictions = model.predict(test_x)

print("Model trained and used to predict on the test set.")

## Step 5 — the confusion matrix: the four kinds of right and wrong

The confusion matrix counts the four outcomes. Rows are the true class, columns
are what the model predicted:

- top-left: legit correctly cleared (a win)
- top-right: legit wrongly flagged as fraud (a false alarm)
- bottom-left: fraud missed and waved through (the dangerous one)
- bottom-right: fraud correctly caught (a win)

In [ ]:
from sklearn.metrics import confusion_matrix   # the four-count table

matrix = confusion_matrix(test_label, predictions)

print("Confusion matrix:")
print(matrix)
print()

legit_cleared = matrix[0, 0]
false_alarms  = matrix[0, 1]
frauds_missed = matrix[1, 0]
frauds_caught = matrix[1, 1]

print("legit correctly cleared:", legit_cleared)
print("false alarms           :", false_alarms)
print("frauds MISSED          :", frauds_missed)
print("frauds caught          :", frauds_caught)

## Step 6 — accuracy, precision, recall, F1

Now the real scores for the trained model:

- **accuracy**: fraction of all predictions that were right
- **precision**: of the transactions we flagged as fraud, how many really were?
- **recall**: of the real frauds, how many did we catch?
- **F1**: a single number that is high only when precision and recall are both
  high

In [ ]:
from sklearn.metrics import accuracy_score, precision_score
from sklearn.metrics import recall_score, f1_score

accuracy = accuracy_score(test_label, predictions)
precision = precision_score(test_label, predictions, zero_division=0)
recall = recall_score(test_label, predictions, zero_division=0)
f1 = f1_score(test_label, predictions, zero_division=0)

print("accuracy :", round(accuracy, 3), " (looks high - remember the paradox)")
print("precision:", round(precision, 3), " (of our alarms, how many were real)")
print("recall   :", round(recall, 3), " (of all real frauds, how many we caught)")
print("F1       :", round(f1, 3), " (balance of precision and recall)")

## Step 7 — ROC-AUC: one number for ranking quality

The model gives each transaction a probability of being fraud. A good model
gives higher probabilities to real frauds than to legit ones. **ROC-AUC**
measures exactly that ranking quality: 1.0 is perfect, 0.5 is no better than
guessing. It does not depend on where you set the decision line, so it is a good
single summary of the model itself.

In [ ]:
from sklearn.metrics import roc_auc_score

# We need the predicted PROBABILITY of fraud, not the hard yes/no call.
probability_of_fraud = model.predict_proba(test_x)[:, 1]

roc_auc = roc_auc_score(test_label, probability_of_fraud)
print("ROC-AUC:", round(roc_auc, 3), " (1.0 is perfect, 0.5 is random guessing)")

## Step 8 — moving the decision line (the threshold)

By default the model flags fraud when its probability is at least **0.5**. But
0.5 is just a convention. If we **lower** the cut-off we catch more fraud
(recall goes up) but raise more false alarms (precision goes down). If we
**raise** it, the opposite. Let us sweep several cut-offs and watch the two
scores pull apart.

In [ ]:
thresholds_to_try = [0.8, 0.6, 0.5, 0.4, 0.3, 0.2]

print("threshold | precision | recall | frauds caught")
for threshold in thresholds_to_try:
    prediction_at_threshold = (probability_of_fraud >= threshold).astype(int)

    precision_here = precision_score(test_label, prediction_at_threshold,
                                     zero_division=0)
    recall_here = recall_score(test_label, prediction_at_threshold,
                               zero_division=0)
    caught = int(np.sum((prediction_at_threshold == 1) & (test_label == 1)))

    print(f"   {threshold:.1f}    |   {precision_here:.3f}   | "
          f"{recall_here:.3f}  |     {caught}")

print()
print("Lower cut-off -> catch more fraud (recall up), more false alarms (precision down).")

## Step 9 — picture the trade-off

The same idea as a graph. As we sweep the cut-off, precision and recall move in
opposite directions. `precision_recall_curve` works them out at many cut-offs
for us, so we can see the whole trade-off in one line.

In [ ]:
from sklearn.metrics import precision_recall_curve

precision_values, recall_values, curve_thresholds = precision_recall_curve(
    test_label, probability_of_fraud)

plt.figure(figsize=(7, 5))
plt.plot(recall_values, precision_values, color="#2E75B6", linewidth=3)
plt.xlabel("recall  (fraction of real frauds caught)")
plt.ylabel("precision  (fraction of fraud alarms that were real)")
plt.title("Precision falls as we push recall up - a trade-off")
plt.show()

## Step 10 — the same ruler on real data: breast cancer

Fraud is synthetic; this step is real. We load a breast-cancer dataset (built
into scikit-learn) where **0 = malignant** and **1 = benign**. Missing a
malignant tumour is the dangerous mistake, so here recall — "of the real
malignancies, how many did we catch?" — is the number that matters most. We fit
the same logistic regression and sweep the threshold exactly as above.

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
cancer_x, cancer_y = cancer.data, 1 - cancer.target   # 1 = malignant (the one we must catch)

cx_train, cx_test, cy_train, cy_test = train_test_split(
    cancer_x, cancer_y, test_size=0.3, random_state=42)

cancer_model = LogisticRegression(max_iter=5000)
cancer_model.fit(cx_train, cy_train)
probability_malignant = cancer_model.predict_proba(cx_test)[:, 1]

print("cancer accuracy:", round(accuracy_score(cy_test, cancer_model.predict(cx_test)), 3))
print("cancer ROC-AUC :", round(roc_auc_score(cy_test, probability_malignant), 3))
print("malignant cases in test:", int(np.sum(cy_test == 1)))
print()
print("threshold | precision | recall | malignancies caught")
for threshold in [0.9, 0.7, 0.5, 0.3]:
    pt = (probability_malignant >= threshold).astype(int)
    p = precision_score(cy_test, pt, zero_division=0)
    r = recall_score(cy_test, pt, zero_division=0)
    caught = int(np.sum((pt == 1) & (cy_test == 1)))
    print(f"   {threshold:.1f}    |   {p:.3f}   | {r:.3f}  |     {caught}")

print()
print("A medical call: how many false alarms is catching one more tumour worth?")

## What you just did

You saw the **accuracy paradox**: on imbalanced data a lazy model can look 95%
accurate and be useless. You then judged a real classifier honestly with the
**confusion matrix, precision, recall, F1 and ROC-AUC**, and moved the
**threshold** to trade catching more cases against raising more false alarms —
on synthetic fraud data and on real breast-cancer data. Choosing that cut-off is
a business (or medical) decision: how bad is a missed case compared with a false
alarm? That judgement, not the algorithm, is the heart of classification in
industry.

The Stretch cells below are optional extras for the confident.

### Stretch (optional) — precision and recall by hand

Skip this if you are new to code. If you want to see there is no magic in the
metrics: precision and recall are just the four confusion-matrix counts divided.
We rebuild them from `frauds_caught`, `false_alarms` and `frauds_missed` and
check they match the values `scikit-learn` gave in Step 6.

In [ ]:
precision_by_hand = (frauds_caught / (frauds_caught + false_alarms)
                    if (frauds_caught + false_alarms) > 0 else 0.0)
recall_by_hand = (frauds_caught / (frauds_caught + frauds_missed)
                if (frauds_caught + frauds_missed) > 0 else 0.0)

print("precision by hand   :", round(precision_by_hand, 3))
print("precision (sklearn) :", round(precision, 3))
print()
print("recall by hand      :", round(recall_by_hand, 3))
print("recall (sklearn)    :", round(recall, 3))
print("\nSame numbers - the metrics are just the four counts, divided.")

### Stretch (optional) — one lever for imbalance: class weights

Skip this unless you want a fix. The model above quietly cared about every
mistake equally, so with fraud so rare it barely bothered to catch it. Adding
`class_weight="balanced"` tells the model a mistake on the rare class costs
more. We retrain with it and compare recall. Notice recall usually jumps: we
catch far more fraud, at the price of more false alarms. Which trade you want is,
again, a business call.

In [ ]:
# Same model, but now a fraud mistake is weighted as more costly.
weighted_model = LogisticRegression(max_iter=1000, class_weight="balanced")
weighted_model.fit(train_x, train_label)
weighted_predictions = weighted_model.predict(test_x)

recall_plain = recall_score(test_label, predictions, zero_division=0)
recall_weighted = recall_score(test_label, weighted_predictions, zero_division=0)
precision_plain = precision_score(test_label, predictions, zero_division=0)
precision_weighted = precision_score(test_label, weighted_predictions,
                                     zero_division=0)

print("plain model    -> recall:", round(recall_plain, 3),
      " precision:", round(precision_plain, 3))
print("weighted model -> recall:", round(recall_weighted, 3),
      " precision:", round(precision_weighted, 3))
print("\nWeighting the rare class usually lifts recall (more fraud caught)")
print("at the cost of some precision (more false alarms).")

## Your turn (5 minutes)

Small changes, so the ideas stick. Copy a line from above into the empty cell
and edit it.

1. Change the fraud dataset's `weights` from `[0.95, 0.05]` to `[0.99, 0.01]`
   and re-run Steps 1–6. Does the accuracy trap get worse? Does recall drop?
2. On the breast-cancer data, raise the threshold to `0.9` and note precision
   and recall. Would you approve that threshold for a screening test? Why?
3. In one sentence in a text cell: when would you rather have high precision,
   and when would you rather have high recall?

In [ ]:
# Your turn — write your code here.